In [5]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go

# Get datasets

In [6]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)

In [7]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

# Get Box Plots

In [8]:
def print_box_plot(df, threshold_value, cat, file_name):

    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['true_label'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    fig = px.box(#scatter
        df, 
        y="distance", 
        x="x_offset", 
        color="true_label", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Add the threshold line
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(cat_range) - 0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color="red", dash="dash"),
        xref="x",
        yref="y",
    )
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        #scattergap=0.75,
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value",
        boxgroupgap=0, 
        boxgap=0
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, "RQ2", f'results/RQ2/', file_name, yaxis_range=[0, 1])


In [9]:
m = "T"
metric = "trace"
threshold = "N"
columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 

for hw in c.hardware:
    threshold_value = c.getModelTolerance(hw)[metric]
    df_hw = df[df['hardware'] == hw]
    df_metric = df_hw[df_hw['metric'] == m]
    df_threshold = df_metric[df_metric['threshold'] == threshold]
    
    for key, categories in c.table_data.items():
        selected_columns = df_threshold[[key, 'true_label', 'distance']]
        file_name = f'{hw}_{threshold}_{key}'
        print_box_plot(selected_columns, threshold_value, key, file_name)
        
    for cat in columns: 
        min_val = int(df_equiv[cat].min())
        max_val = int(df_equiv[cat].max())
        cat_range = list(map(int, range(min_val, max_val + 1)))
        selected_columns = df_threshold[[cat, 'true_label', 'distance']]  
        file_name = f'{hw}_{threshold}_{cat}'
        print_box_plot(selected_columns, threshold_value, cat, file_name)

# Subplots

In [ ]:
def print_scatter_plot(df, threshold_value, cat, file_name, subplot_idx, fig):
    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['true_label'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the scatter plot with the adjusted x-axis values
    scatter = go.Scatter(
        y=df['distance'],
        x=df['x_offset'],
        mode='markers',
        marker=dict(color=df['true_label'].map({"Equivalent mutant": 'blue', "Non-Equivalent mutant": 'red'}), size=10),
        name=cat
    )
    
    #scatter = px.scatter(
    #    df, 
    #    y="distance", 
     #   x="x_offset", 
     #   color="true_label", 
     #   category_orders={cat: cat_range}
   # )
    
    # Add the threshold line
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(cat_range) - 0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color="red", dash="dash"),
        xref="x",
        yref="y",
    )
    
    # Add the scatter plot to the subplot
    fig.add_trace(scatter, row=1, col=subplot_idx)


In [ ]:
def create_subplots_for_hardware(hw, threshold_value, table_data, df, m, threshold):
    # Create the subplot figure with enough subplots for each category
    fig = sp.make_subplots(
        rows=1,
        cols=len(table_data),  # Number of categories as columns
        shared_yaxes=True,  # Same y-axis for better comparison
        subplot_titles=list(table_data.keys()),  # Titles as the keys of table_data
        horizontal_spacing=0.1  # Adjust space between subplots
    )
    
    df_metric = df[df['metric'] == m]
    df_threshold = df_metric[df_metric['threshold'] == threshold]
    
    # Loop over each category in table_data and add it as a subplot
    for subplot_idx, (key, categories) in enumerate(table_data.items(), start=1):
        selected_columns = df_threshold[[key, 'true_label', 'distance']]
        file_name = f'{hw}_{threshold}_{key}'
        print_scatter_plot(selected_columns, threshold_value, key, file_name, subplot_idx, fig)    
        
    # Adjust layout for better visualization
    fig.update_layout(
        title_text=f"{hw} Performance Metrics",
        height=600,
        width=2000,
        showlegend=True,
        xaxis_title="Characteristic",
        yaxis_title="Distance between original and mutant"
    )
    
    # Save the figure for this hardware
    c.setup_layout_and_save(fig, f"{hw} Performance Metrics", 'results/RQ2/hardware', f"{hw}_{threshold}_metrics", yaxis_range=[0, 1])
    #setup_layout_and_save(fig, "RQ2", f'results/RQ2/', file_name, yaxis_range=[0, 1])


In [ ]:
# Example of usage for each hardware
m = "F"
metric = "fidelity"
threshold = "0.1"
threshold_value = c.getModelTolerance("kyiv")[metric]  # This can be adjusted based on your model

for hw in ["kyiv"]: #hardware:
    create_subplots_for_hardware(hw, threshold_value, c.table_data, df, m, threshold)

In [ ]:
# Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        scattergap=0.75,

        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),

        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text="Expected value"
    )